In [1]:
from kafka import KafkaConsumer

server = 'localhost:9092' #same server and topic as producer
topic_name = 'rides'

In [2]:
from models import Ride, ride_deserializer

In [3]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-database',
    value_deserializer=ride_deserializer
)

In [4]:
next(consumer)

ConsumerRecord(topic='rides', partition=0, leader_epoch=1, offset=0, timestamp=1773612649828, timestamp_type=0, key=None, value=Ride(PULocationID=142, DOLocationID=237, trip_distance=2.28, total_amount=24.94, tpep_pickup_datetime=1761958147000), headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=127, serialized_header_size=-1)

In [5]:
next(consumer)

ConsumerRecord(topic='rides', partition=0, leader_epoch=1, offset=1, timestamp=1773613398643, timestamp_type=0, key=None, value=Ride(PULocationID=142, DOLocationID=237, trip_distance=2.28, total_amount=24.94, tpep_pickup_datetime=1761958147000), headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=127, serialized_header_size=-1)

In [6]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)

conn.autocommit = True
cur = conn.cursor()

In [7]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
           VALUES (%s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID,
         ride.trip_distance, ride.total_amount, pickup_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to rides and writing to PostgreSQL...
Inserted 100 rows...
Inserted 200 rows...
Inserted 300 rows...
Inserted 400 rows...
Inserted 500 rows...
Inserted 600 rows...
Inserted 700 rows...
Inserted 800 rows...
Inserted 900 rows...
Inserted 1000 rows...


KeyboardInterrupt: 

In [ ]:
# the code above will keep listening to the stream, waiting for next value in consumer (our producer has 1000 vaues we can see all got stored)